In [2]:
"""
Summaries of the Wildfire Risk to Communities, Housing Unit Risk
Maxwell.Cook@colostate.edu
"""

import os, sys
import rasterio as rio

from rasterstats import zonal_stats

# import the __functions.py (custom functions)
sys.path.append(os.getcwd()) # add code folder to system path
from __functions import *  # imports all custom functions

# local data directories
datadir = '/Users/mcc/Library/CloudStorage/Box-Box/MCC/data/'
projdir = os.path.dirname(os.getcwd())
print(projdir)

print("Ready !")

/Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation
Ready !


In [3]:
# load and prepare the incidents and perimeter data

# read in the incidents data (gathered from the ICS209+ by Evan Herpe et al)
# this subset is "timber only" fires
incis = os.path.join(projdir, 'data/spatial/mod/ics209plus_2014to2023_perimeters_tcc.gpkg')
incis = gpd.read_file(incis)

# --- Keep only forested incidents
incis = incis.copy()
incis = incis[
    (incis['FUEL_MODEL'].str.contains('Timber', case=False)) |
    (incis['tcc']>=10) # keep forested incidents
]

print(f"{len(incis)} incidents in the subset (timber fires).")

4259 incidents in the subset (timber fires).


In [5]:
# load fire perimeters and intersect with counties
# calculate the intersecting counties and overlap area
counties = os.path.join(datadir,"boundaries/political/TIGER/tl_2024_us_county/tl_2024_us_county.shp")
counties = gpd.read_file(counties).to_crs(incis.crs)

# Intersect with fire perimeters
fire_fips = gpd.overlay(incis, counties[['GEOID', 'geometry']], how="intersection", keep_geom_type=True)
fire_fips['overlap_acres'] = fire_fips.geometry.area / 4046.86  # calculate the area in acres

# calculate the proportion of burned area in each county
fire_fips['county_prop'] = round(fire_fips['overlap_acres'] / fire_fips['FINAL_ACRES'], 2)
fire_fips[['INCIDENT_ID','GEOID','overlap_acres','county_prop']].head()

,INCIDENT_ID,GEOID,overlap_acres,county_prop
0,2015_2845771_COPENHAGEN,02070,3207.580898,0.96
1,2022_14468677_PAULS CREEK,02164,17904.022678,1.00
2,2019_10725960_LEVELOCK,02164,10370.645117,1.21
3,2022_14498802_IOWITHLA RIVER,02070,43424.665660,1.01
4,2015_2854934_LITTLE KOKWOK,02070,2233.845128,1.16


In [6]:
# load the WRC data downloaded from the USFS Research Data Archive
hurisk_fp = list_files(datadir, 'HURisk_CONUS.tif', recursive=True)[0]
popden_fp = list_files(datadir, 'PopDen_CONUS.tif', recursive=True)[0]

# find the counties overlapping fire data
counties_fire = counties[counties['GEOID'].isin(fire_fips['GEOID'].unique())]

# now calculate the mean HURisk and PopDen for each county:

# Load the HURisk raster and mask values > 0
with rio.open(hurisk_fp) as src:
    hurisk = src.read(1)        # read band 1
    transform = src.transform
    nodata = src.nodata

# Mask invalid or unwanted values
hurisk = hurisk.astype(float)
hurisk[hurisk == nodata] = np.nan
hurisk[hurisk <= 0] = np.nan  # keep only positive (nonzero) values

# Now pass to zonal_stats
county_stats = pd.DataFrame(zonal_stats(
    vectors=counties_fire,
    raster=hurisk,
    affine=transform,
    stats=['mean'],   # this will now calculate mean of >0 values
    nodata=np.nan     # make sure NaNs are excluded
))

county_stats['GEOID'] = counties_fire['GEOID'].values
county_stats = county_stats.rename(columns={'mean': 'WRC_HURisk'})

# Repeat for PopDen
popden_stats = pd.DataFrame(zonal_stats(
    counties_fire,
    popden_fp,
    stats=['mean'],
    geojson_out=False
))
popden_stats['GEOID'] = counties_fire['GEOID'].values
popden_stats = popden_stats.rename(columns={'mean': 'WRC_PopDen'})

# Join them together
county_summary = county_stats.merge(popden_stats, on='GEOID')

# save a county shapefile:
counties = counties.merge(county_summary, on='GEOID')
out_fp = os.path.join(projdir,'data/spatial/mod/WRC/counties_hurisk_popden.gpkg')
os.makedirs(os.path.dirname(out_fp), exist_ok=True)
counties.to_file(out_fp)

county_summary.head()

,WRC_HURisk,GEOID,WRC_PopDen
0,23816.520128,06091,1.446153
1,191.314229,21053,19.963024
2,1741.924384,01027,9.639502
3,2132.140363,41063,0.947596
4,182.778710,08109,0.834110


In [7]:
# join county-level risk/population summaries to the fire-county overlaps
fire_fips_c = fire_fips.merge(county_summary, on='GEOID', how='left')

# calculate the weighted average by fire
weighted_summary = (
    fire_fips_c
    .groupby('INCIDENT_ID')
    .apply(lambda g: pd.Series({
        'WRC_HURisk_wt': (g['WRC_HURisk'] * g['county_prop']).sum(),
        'WRC_PopDen_wt': (g['WRC_PopDen'] * g['county_prop']).sum()
    }), include_groups=False)
    .reset_index()
)

# check the results
weighted_summary.head()

,INCIDENT_ID,WRC_HURisk_wt,WRC_PopDen_wt
0,2014_1075636_DOG ROCK,10723.417907,4.847122
1,2014_1077684_APPLEGATE,12182.648432,111.981102
2,2014_1126016_HIGHWAY 613 FIRE,7661.515480,68.373597
3,2014_1137798_135A,8326.682050,5.712777
4,2014_1138291_WEST RANGE FIRE,1489.296888,1.012131


In [8]:
# save the table
out_fp = os.path.join(projdir,'data/tabular/WRC_hurisk_popden_weighted.csv')
weighted_summary.to_csv(out_fp)
print(f"Saved to {out_fp}")

Saved to /Users/mcc/Library/CloudStorage/Box-Box/MCC/projects/ReSHAPE/valuation/data/tabular/WRC_hurisk_popden_weighted.csv


In [9]:
len(weighted_summary)

4259